# Kapitel 5: Was hat die Maschine gelernt?

## Die Reise bis hierher

- **Kapitel 1** — Aus 3,6 Mio. echten Amazon-Bewertungen zogen wir 400.000
- **Kapitel 2** — Wir entdeckten die natürliche Verteilung und balancierten sie
- **Kapitel 3** — Wörter wurden zu TF-IDF-Vektoren
- **Kapitel 4** — Logistic Regression lernte, Emotionen zu erkennen

Jetzt erzählen wir die Geschichte visuell.

## 5.1 Setup

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Visualisierung") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df_clean = spark.read.parquet("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/cleaned_reviews.parquet")
df_preds = spark.read.parquet("/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/predictions.parquet")

print(f"Bereinigte Daten: {df_clean.count():,}")
print(f"Vorhersagen:      {df_preds.count():,}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# ═══ ELEGANT DARK THEME ═══
BG      = '#0E1117'
CARD    = '#1A1F2E'
TEXT    = '#E2E8F0'
MUTED   = '#64748B'
GRID    = '#2D3548'
TEAL    = '#14B8A6'
TEAL_D  = '#0D9488'
RED     = '#EF4444'
RED_S   = '#FCA5A5'
GREEN   = '#10B981'
GREEN_S = '#6EE7B7'
BLUE    = '#3B82F6'
AMBER   = '#F59E0B'
PURPLE  = '#A78BFA'
ORANGE  = '#F5B041'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'text.color': TEXT, 'axes.labelcolor': TEXT,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': GRID, 'grid.color': GRID,
    'grid.alpha': 0.4, 'grid.linewidth': 0.5,
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 16, 'axes.titleweight': 'bold',
    'axes.titlepad': 18, 'axes.labelpad': 10, 'axes.labelsize': 12,
})

OUT = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output"
def savefig(fig, name):
    fig.savefig(f"{OUT}/{name}.png", dpi=200, bbox_inches='tight', facecolor=BG)

print("Theme geladen.")

## 5.2 Sentiment-Verteilung

In [ ]:
from pyspark.sql.functions import col, count as spark_count

label_dist = df_clean.groupBy("label").agg(spark_count("*").alias("count")).orderBy("label").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [1.3, 1]})

bars = axes[0].bar(["Negativ", "Positiv"], label_dist["count"],
                   color=[RED, GREEN], width=0.55, edgecolor='none', zorder=3)
axes[0].set_ylim(0, label_dist["count"].max() * 1.18)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., h + 2000,
                f'{int(h):,}', ha='center', va='bottom',
                fontsize=16, fontweight='bold', color=TEXT)
axes[0].set_title("Sentiment-Verteilung")
axes[0].set_ylabel("Anzahl")
axes[0].grid(axis='y', alpha=0.3); axes[0].set_axisbelow(True)
for spine in axes[0].spines.values(): spine.set_visible(False)

wedges, texts, autotexts = axes[1].pie(
    label_dist["count"], labels=["Negativ", "Positiv"],
    colors=[RED, GREEN], autopct='%1.1f%%', startangle=90,
    wedgeprops={'linewidth': 3, 'edgecolor': BG, 'width': 0.65},
    textprops={'color': TEXT, 'fontsize': 13}, pctdistance=0.75)
for t in autotexts: t.set_fontsize(14); t.set_fontweight('bold'); t.set_color('white')
axes[1].set_title("Anteil")

plt.tight_layout(pad=2.0)
savefig(fig, '01_sentiment_verteilung')
plt.show()

## 5.3 Textlängen nach Sentiment

In [ ]:
from pyspark.sql.functions import length

df_len = df_clean.withColumn("text_length", length(col("text")))
pos_len = df_len.filter(col("label")==1).select("text_length").toPandas()
neg_len = df_len.filter(col("label")==0).select("text_length").toPandas()

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.hist(neg_len["text_length"].dropna(), bins=50, alpha=0.55, color=RED,
        edgecolor='none', label="Negativ", range=(0, 2000), zorder=3)
ax.hist(pos_len["text_length"].dropna(), bins=50, alpha=0.55, color=GREEN,
        edgecolor='none', label="Positiv", range=(0, 2000), zorder=3)

neg_med = neg_len["text_length"].median()
pos_med = pos_len["text_length"].median()
ax.axvline(neg_med, color=RED_S, linestyle='--', linewidth=1.5, alpha=0.8, label=f'Median neg. ({neg_med:.0f})')
ax.axvline(pos_med, color=GREEN_S, linestyle='--', linewidth=1.5, alpha=0.8, label=f'Median pos. ({pos_med:.0f})')

ax.set_xlabel("Textlänge (Zeichen)"); ax.set_ylabel("Anzahl")
ax.set_title("Textlängen nach Sentiment")
ax.legend(frameon=False, fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
savefig(fig, '02_textlaenge_sentiment')
plt.show()

## 5.4 Diskriminative Wörter

In Kapitel 3 haben wir die häufigsten Wörter gesehen. Hier gehen wir einen Schritt weiter:
Welche Wörter sind am **stärksten unterscheidend** zwischen positiv und negativ?
Wir berechnen das Pos/Neg-Verhältnis — Wörter mit hohem Ratio sind typisch positiv,
Wörter mit niedrigem Ratio typisch negativ.

In [ ]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from pyspark.sql.functions import explode

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
custom_stopwords = StopWordsRemover().getStopWords() + [
    "one", "get", "time", "really", "even", "also"
]
remover = StopWordsRemover(inputCol="tokens", outputCol="filtered", stopWords=custom_stopwords)
df_filt = remover.transform(tokenizer.transform(df_clean))
df_words = df_filt.select("label", explode("filtered").alias("word"))

# Pos/Neg counts pro Wort
pos_counts = df_words.filter(col("label")==1).groupBy("word").agg(spark_count("*").alias("pos"))
neg_counts = df_words.filter(col("label")==0).groupBy("word").agg(spark_count("*").alias("neg"))

# Join und Mindest-Häufigkeit filtern
df_join = pos_counts.join(neg_counts, "word").filter((col("pos") + col("neg")) > 50)

# Ratio berechnen
df_ratio = df_join.withColumn("ratio", col("pos") / (col("neg") + 1))

top_pos_disc = df_ratio.orderBy(col("ratio").desc()).limit(20).toPandas()
top_neg_disc = df_ratio.orderBy(col("ratio").asc()).limit(20).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 8))

# Positiv
axes[0].barh(top_pos_disc["word"][::-1], top_pos_disc["ratio"][::-1],
             color=GREEN, edgecolor='none', height=0.7, zorder=3)
for i, (w, r) in enumerate(zip(top_pos_disc["word"][::-1], top_pos_disc["ratio"][::-1])):
    axes[0].text(r + 0.1, i, f'{r:.1f}x', va='center', fontsize=9, color=MUTED)
axes[0].set_title("✅  Top 20 — Typisch Positiv", color=GREEN)
axes[0].set_xlabel("Pos/Neg Verhältnis")
axes[0].grid(axis='x', alpha=0.3); axes[0].set_axisbelow(True)
for spine in axes[0].spines.values(): spine.set_visible(False)

# Negativ
axes[1].barh(top_neg_disc["word"][::-1], top_neg_disc["ratio"][::-1],
             color=RED, edgecolor='none', height=0.7, zorder=3)
for i, (w, r) in enumerate(zip(top_neg_disc["word"][::-1], top_neg_disc["ratio"][::-1])):
    axes[1].text(r + 0.01, i, f'{r:.2f}x', va='center', fontsize=9, color=MUTED)
axes[1].set_title("❌  Top 20 — Typisch Negativ", color=RED)
axes[1].set_xlabel("Pos/Neg Verhältnis")
axes[1].grid(axis='x', alpha=0.3); axes[1].set_axisbelow(True)
for spine in axes[1].spines.values(): spine.set_visible(False)

plt.tight_layout(pad=2.0)
savefig(fig, '03_diskriminative_woerter')
plt.show()

## 5.5 Word Cloud

In [ ]:
from wordcloud import WordCloud

# Häufigkeiten für Word Cloud (absolute counts statt ratio)
top_pos_freq = df_words.filter(col("label")==1).groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(80).toPandas()
top_neg_freq = df_words.filter(col("label")==0).groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(80).toPandas()

pos_freq = dict(zip(top_pos_freq["word"], top_pos_freq["count"]))
neg_freq = dict(zip(top_neg_freq["word"], top_neg_freq["count"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

wc_pos = WordCloud(width=900, height=450, background_color=CARD,
                   colormap='Greens', max_words=80, prefer_horizontal=0.8,
                   contour_width=0, margin=15).generate_from_frequencies(pos_freq)
axes[0].imshow(wc_pos, interpolation="bilinear")
axes[0].set_title("POSITIV", fontsize=18, fontweight='bold', color=GREEN, pad=12)
axes[0].axis("off")

wc_neg = WordCloud(width=900, height=450, background_color=CARD,
                   colormap='Reds', max_words=80, prefer_horizontal=0.8,
                   contour_width=0, margin=15).generate_from_frequencies(neg_freq)
axes[1].imshow(wc_neg, interpolation="bilinear")
axes[1].set_title("NEGATIV", fontsize=18, fontweight='bold', color=RED, pad=12)
axes[1].axis("off")

plt.tight_layout(pad=1.5)
savefig(fig, '04_wordcloud')
plt.show()

## 5.6 Confusion Matrix

In [ ]:
cm_data = df_preds.groupBy("label","prediction").count().orderBy("label","prediction").toPandas()
cm = np.zeros((2,2), dtype=int)
for _, row in cm_data.iterrows():
    cm[int(row["label"]), int(row["prediction"])] = int(row["count"])
cm_pct = cm / cm.sum() * 100

cmap = LinearSegmentedColormap.from_list('teal', [CARD, TEAL_D, TEAL], N=256)
labels = ["Negativ (0)", "Positiv (1)"]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for idx, (data, fmt, title) in enumerate([(cm, "{:,}", "Absolute Werte"), (cm_pct, "{:.1f}%", "Prozentual")]):
    im = axes[idx].imshow(data, cmap=cmap, aspect='auto')
    axes[idx].set_xticks([0,1]); axes[idx].set_yticks([0,1])
    axes[idx].set_xticklabels(labels, fontsize=12)
    axes[idx].set_yticklabels(labels, fontsize=12)
    axes[idx].set_xlabel("Vorhersage", fontsize=13)
    axes[idx].set_ylabel("Tatsächlich", fontsize=13)
    axes[idx].set_title(f"Confusion Matrix — {title}", fontsize=15)
    for i in range(2):
        for j in range(2):
            axes[idx].text(j, i, fmt.format(data[i,j]),
                          ha="center", va="center", fontsize=20,
                          fontweight="bold", color='white')
    for spine in axes[idx].spines.values(): spine.set_visible(False)

plt.tight_layout(pad=2.0)
savefig(fig, '05_confusion_matrix')
plt.show()

## 5.7 ROC-Kurve

In [ ]:
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc

# Probability extrahieren mit vector_to_array
roc_pd = df_preds.withColumn(
    "prob_positive", vector_to_array(col("probability"))[1]
).select("label", "prob_positive").toPandas()

fpr, tpr, _ = roc_curve(roc_pd["label"], roc_pd["prob_positive"])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 7))
ax.fill_between(fpr, tpr, alpha=0.12, color=TEAL, zorder=2)
ax.plot(fpr, tpr, color=TEAL, lw=2.5, label=f"Logistic Regression (AUC = {roc_auc:.4f})", zorder=3)
ax.plot([0,1], [0,1], color=MUTED, lw=1.2, linestyle="--", label="Zufall (AUC = 0.5)", zorder=2)
ax.text(0.55, 0.25, f'AUC = {roc_auc:.4f}', fontsize=28, fontweight='bold',
        color=TEAL, alpha=0.3, ha='center')

ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC-Kurve")
ax.legend(frameon=False, fontsize=11, loc="lower right")
ax.grid(True, alpha=0.3); ax.set_axisbelow(True)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
savefig(fig, '06_roc_kurve')
plt.show()

## 5.8 Wahrscheinlichkeits-Verteilung

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))

ax.hist(roc_pd[roc_pd["label"]==0]["prob_positive"], bins=50, alpha=0.6,
        color=RED, edgecolor='none', label="Tatsächlich Negativ", zorder=3)
ax.hist(roc_pd[roc_pd["label"]==1]["prob_positive"], bins=50, alpha=0.6,
        color=GREEN, edgecolor='none', label="Tatsächlich Positiv", zorder=3)
ax.axvline(x=0.5, color='white', linestyle="--", linewidth=1.5, label="Schwellenwert", zorder=4)

ylim = ax.get_ylim()
ax.text(0.15, ylim[1]*0.85, '← NEGATIV', fontsize=14,
        color=RED_S, fontweight='bold', alpha=0.6, ha='center')
ax.text(0.85, ylim[1]*0.85, 'POSITIV →', fontsize=14,
        color=GREEN_S, fontweight='bold', alpha=0.6, ha='center')

ax.set_xlabel("Wahrscheinlichkeit (positiv)"); ax.set_ylabel("Anzahl")
ax.set_title("Wie sicher ist das Modell?")
ax.legend(frameon=False, fontsize=11)
ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
savefig(fig, '08_wahrscheinlichkeiten')
plt.show()

## 5.9 Metriken-Dashboard

In [ ]:
total_p = cm.sum()
acc = (cm[0,0]+cm[1,1])/total_p
prec_pos = cm[1,1]/(cm[0,1]+cm[1,1]) if (cm[0,1]+cm[1,1])>0 else 0
rec_pos = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
f1_v = 2*prec_pos*rec_pos/(prec_pos+rec_pos) if (prec_pos+rec_pos)>0 else 0

metrics = {
    'Accuracy': (acc, TEAL), 'AUC-ROC': (roc_auc, BLUE),
    'Precision': (prec_pos, GREEN), 'Recall': (rec_pos, AMBER),
    'F1-Score': (f1_v, PURPLE)
}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(metrics.keys())[::-1]
values = [metrics[n][0] for n in names]
colors = [metrics[n][1] for n in names]

bars = ax.barh(names, values, color=colors, height=0.55, edgecolor='none', zorder=3)
for bar, val in zip(bars, values):
    ax.text(val + 0.015, bar.get_y() + bar.get_height()/2.,
            f'{val:.4f}', ha='left', va='center', fontsize=14, fontweight='bold', color=TEXT)

ax.set_xlim([0, 1.12])
ax.set_title("Evaluations-Metriken")
ax.axvline(x=0.5, color=MUTED, linestyle="--", alpha=0.4, zorder=1)
ax.axvline(x=0.8, color=MUTED, linestyle=":", alpha=0.3, zorder=1)
ax.grid(axis='x', alpha=0.2); ax.set_axisbelow(True)
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout()
savefig(fig, '07_metriken_dashboard')
plt.show()

## 5.10 Einzelne Bewertungen: Was sagt das Modell?

Bisher haben wir nur Statistiken gesehen. Jetzt schauen wir uns
**konkrete Bewertungen** an — richtige und falsche Vorhersagen.

In [ ]:
from pyspark.ml.functions import vector_to_array

df_with_prob = df_preds.withColumn(
    "prob_positive", vector_to_array(col("probability"))[1]
)

examples_pos = df_with_prob.filter(
    (col("label") == 1) & (col("prediction") == 1) & (col("prob_positive") > 0.85)
).withColumn("tlen", length("text")).filter("tlen BETWEEN 80 AND 300") \
 .select("text", "prob_positive", "label", "prediction").limit(5).toPandas()

examples_neg = df_with_prob.filter(
    (col("label") == 0) & (col("prediction") == 0) & (col("prob_positive") < 0.15)
).withColumn("tlen", length("text")).filter("tlen BETWEEN 80 AND 300") \
 .select("text", "prob_positive", "label", "prediction").limit(5).toPandas()

examples_wrong = df_with_prob.filter(
    col("label") != col("prediction")
).withColumn("tlen", length("text")).filter("tlen BETWEEN 80 AND 300") \
 .select("text", "prob_positive", "label", "prediction").limit(6).toPandas()

examples_pos["typ"] = "Richtig Positiv"
examples_neg["typ"] = "Richtig Negativ"
examples_wrong["typ"] = "Falsch"

examples = pd.concat([examples_neg, examples_wrong, examples_pos]).reset_index(drop=True)
examples["kurztext"] = examples["text"].apply(lambda x: x[:80] + "..." if len(x) > 80 else x)

fig, ax = plt.subplots(figsize=(14, 9))

farben = []
for t in examples["typ"]:
    if t == "Richtig Positiv": farben.append(GREEN)
    elif t == "Richtig Negativ": farben.append(RED)
    else: farben.append(ORANGE)

ax.barh(range(len(examples)), examples["prob_positive"],
        color=farben, height=0.65, edgecolor='none', zorder=3)

ax.set_yticks(range(len(examples)))
ax.set_yticklabels(examples["kurztext"], fontsize=9, color=TEXT)
ax.set_xlabel("Wahrscheinlichkeit (positiv)", fontsize=12)
ax.set_title("Modellbewertung: Richtige und falsche Vorhersagen", fontsize=16)
ax.axvline(x=0.5, color='white', linestyle="--", linewidth=1.2,
           label="Schwellenwert", alpha=0.7, zorder=4)

ax.text(0.15, -1, '← NEGATIV', fontsize=12, color=RED_S,
        fontweight='bold', alpha=0.5, ha='center')
ax.text(0.85, -1, 'POSITIV →', fontsize=12, color=GREEN_S,
        fontweight='bold', alpha=0.5, ha='center')

ax.invert_yaxis()
ax.grid(axis='x', alpha=0.2); ax.set_axisbelow(True)
for spine in ax.spines.values(): spine.set_visible(False)
ax.legend(frameon=False, fontsize=10, loc='lower right')

plt.tight_layout()
savefig(fig, '09_modellbewertung_beispiele')
plt.show()

### Erkenntnis

Jeder Balken ist eine **echte Amazon-Bewertung**.
- 🟢 **Grün** = korrekt positiv erkannt
- 🔴 **Rot** = korrekt negativ erkannt
- 🟠 **Orange** = falsche Vorhersage

Das Modell erkennt **ganze Ausdrucksmuster** — nicht nur einzelne Wörter.
Fehler treten bei Sarkasmus, gemischten Meinungen und fehlender Kontexterkennung auf.

## 5.11 Gespeicherte Dateien

In [ ]:
import os
output_dir = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output"
print("=" * 55)
print("  GESPEICHERTE DATEIEN")
print("=" * 55)
for f in sorted(os.listdir(output_dir)):
    fp = os.path.join(output_dir, f)
    if os.path.isfile(fp):
        print(f"  {f:45s} {os.path.getsize(fp)/(1024*1024):>6.2f} MB")
    else:
        print(f"  {f:45s} [Ordner]")
print("=" * 55)

## Fazit: Die Antwort auf unsere Frage

### Kann eine Maschine Emotionen lesen?

**Ja — mit Einschränkungen.**

Aus 3,6 Millionen echten Amazon-Bewertungen haben wir eine Stichprobe gezogen,
die natürliche Klassenverteilung analysiert, das Ungleichgewicht korrigiert
und mit TF-IDF + Logistic Regression eine Sentiment-Klassifikation aufgebaut.

### Was das Modell gelernt hat

- *great, love, best, well* → Signale für Zufriedenheit
- *money, bought, product, work* → Signale für Unzufriedenheit

### Der Unterschied zur vorgefertigten Test-Datei

Die `test.csv` auf Kaggle war bereits perfekt balanciert (50/50).
Wir haben bewusst die **echten Trainingsdaten** verwendet, um zu zeigen:
- Reale Daten sind selten perfekt
- Ungleichgewichte müssen erkannt und behandelt werden
- Die Pipeline funktioniert auch unter realistischen Bedingungen

### Mögliche Verbesserungen

| Ansatz | Erwartete Verbesserung |
|--------|----------------------|
| N-Gramme | Kontextbezug: *not good* ≠ *good* |
| Word2Vec | Semantische Ähnlichkeiten |
| Deep Learning (BERT) | Satzstruktur und Kontext |

---

*Vollständig mit Apache Spark und Python — von 3,6 Mio. Rohdaten zur Vorhersage.*